# Meme Reaction V1 — Resumen Completo del Funcionamiento

**Proyecto:** Pipeline automatizado end-to-end para generar videos de "meme reaction" (meme arriba + clip de reacción abajo) y subirlos a redes.

**Ubicación:** `Personal/Drako-Edits/drako-edits/automatizaciones/Meme_Reaction/`

---

## Arquitectura General

El pipeline se ejecuta desde `main.py` que orquesta 8 pasos secuenciales como subprocesses independientes. Usa `.env` del proyecto raíz (`drako-edits/.env`) para credenciales (OpenAI, IG). También existe `auto_meme_reaction.py` — un script legacy que usa `instagrapi` (login API directo) para descargar 1 foto y era el prototipo original.

---

## Pipeline Paso a Paso

### Paso 1: Scraping de Links (`1_scrape_meme_links.py`) ✅
- **Tecnología**: Selenium + Brave (Chromium)
- **Flujo**: Abre navegador visible → pausa para login manual → navega a perfiles target → scrollea capturando shortcodes del DOM en cada scroll (IG virtualiza nodos fuera del viewport)
- **Anti-detección**: Deshabilita flags de automation, simula scrolls humanos (sube/baja)
- **Output**: `historial/links_scrapeados.json` con campos `scrapeados` y `por_descargar`
- **Config**: Perfiles hardcoded (elmello2023), 8 scrolls, delays de 3-5s
- **Nota clave**: IG usa `/reel/` para casi todo (incluso fotos), no se distingue tipo aquí

### Paso 2: Descarga de Memes (`2_download_memes.py`) ✅
- **Tecnología**: instaloader (SIN login, solo queries) + requests + ffmpeg
- **Flujo**: Lee `por_descargar` → para cada shortcode consulta tipo via `Post.from_shortcode()`
- **Comportamiento por tipo**:
  - `GraphImage` → descarga foto directa con requests
  - `GraphVideo` → descarga video con requests, extrae primer frame con ffmpeg, borra video
  - `GraphSidecar` → skip (carousel)
  - `< MIN_LIKES (5000)` → skip
- **Protecciones**: Delay 5s entre posts, pausa de 3 min cada 20 posts, max 50/sesión, auto-stop en rate limit
- **Output**: `memes_descargados/{shortcode}.jpg` + `historial/posts_descargados.json`
- **Bug resuelto**: No usa `instaloader.download_post()` por bug de paths full-width en Windows

### Paso 2.5 (opcional): Revisión Manual (`revisar_memes.py`)
- **Propósito**: Filtrar memes ANTES de gastar tokens de OpenAI
- **Flujo**: Abre cada imagen → usuario decide mantener/desechar → borra los descartados
- **Output**: `historial/descartados_manual.json`

### Paso 3: Clasificación IA (`3_classify_meme.py`) ✅
- **Tecnología**: OpenAI GPT-4o Vision (detail:low, 85 tokens fijos por imagen)
- **Filosofía**: Extraer TODO en 1 sola llamada (imagen ya pagada)
- **Campos extraídos**: validez, es_video_real, categorías (10 posibles), confianza, descripción detallada, ideas_video (2-3 con formato/caption/clip_ideal), background_color, franjas_negras (con porcentajes), dia_especial
- **Categorías**: humor_absurdo, humor_dark, cringe, sad_funny, wholesome, plot_twist, relatable, rage, sus, intellectual
- **Flags**: `--redo`, `--redo-all`, `--max N`
- **Output**: `historial/clasificaciones.json`

### Paso 3.5 (opcional): Review Clasificación (`3.5_review_clasificacion.py`)
- **Propósito**: Validar si la IA clasificó bien (feedback para mejorar prompt)
- **Flujo**: Abre imagen + muestra clasificación → usuario marca OK/MAL con notas
- **Output**: `historial/review_clasificacion.json`

### Paso 4: Match con Clip (`4_match_clip.py`) ✅
- **Tecnología**: OpenAI GPT-4o-mini (texto puro, barato)
- **Modo**: INTERACTIVO — IA sugiere, usuario decide
- **Flujo por meme**:
  1. IA revisa catálogo de clips disponibles (`catalogo_clips.json`)
  2. Elige mejor match con % de accuracy
  3. Sugiere clip ideal (aunque no esté en catálogo)
  4. Sugiere caption (o dice que no necesita)
  5. Usuario puede: aceptar, responder (modo conversación con 'r'), o saltar
- **Modo 'r' (responder)**: Conversación multi-turno donde propones clips y la IA evalúa
- **Output**: `historial/matches.json` (matched + skipped_buscar_clip)

### Paso 5: Verificación IA (`5_verify_match.py`) ❌ NO IMPLEMENTADO
- Enviaría imagen + descripción del clip a IA para confirmar que el match tiene sentido

### Paso 6: Caption IA (`6_generate_caption.py`) ❌ NO IMPLEMENTADO
- Generaría caption final basado en meme + categoría + clip elegido

### Paso 7: Generación de Video (`7_generate_video.py`) ✅
- **Tecnología**: MoviePy 2.0 + Pillow + NumPy
- **Layout**: 1080x1920 (9:16 vertical). Meme arriba (max 70%), clip abajo (max 40%), borde 3%
- **Auto-crop**: Detección programática de barras negras (análisis de pixels, NO depende de IA)
- **Audio**: SIEMPRE del clip (restricción de la automatización)
- **Caption**: Overlay de texto con stroke (tamaños S/M/L/XL configurables)
- **Modo**: Semi-interactivo (confirma cada uno) o `--auto`
- **Flags**: `--redo`, `--redo-all`, `--max`, `--no-crop-meme`
- **Output**: `output/meme_reaction/{shortcode}.mp4` + `configs_generados/{shortcode}.json`

### Paso 8: JSON Config (integrado en Paso 7)
- Ya no es script separado. El paso 7 genera el JSON automáticamente

---

## Scripts Auxiliares

### `catalogar_clips.py` ✅
- **Propósito**: Catalogar clips de reacción para el paso 4
- **Flujo conversacional**: Navega carpetas → abre video → describes → IA mejora descripción → confirmas → asignas categorías → copia a `clips/` → guarda en `catalogo_clips.json`
- **Modelo**: GPT-4o-mini (texto, barato)

### `revisar_memes.py` ✅
- Pre-filtro manual antes de clasificación IA

### `manual_from_config.py` ❌ NO IMPLEMENTADO
- Permitiría re-generar videos con modificaciones (cambiar audio/caption) desde JSON config

### `auto_meme_reaction.py` (Legacy)
- Prototipo usando instagrapi (login API). Solo descarga 1 foto. No se usa en el pipeline modular.

---

## Estructura de Datos

```
historial/
├── links_scrapeados.json       (paso 1: shortcodes + por_descargar)
├── posts_descargados.json      (paso 2: fotos/frames/skipped/errores)
├── descartados_manual.json     (revisar_memes: descartados + mantenidos)
├── clasificaciones.json        (paso 3: clasificados + skipped_video + errores)
├── review_clasificacion.json   (paso 3.5: correctos + incorrectos)
├── matches.json                (paso 4: matched + skipped_buscar_clip)
├── generados.json              (paso 7: generados + errores)

memes_descargados/              (imágenes .jpg por shortcode)
clips/                          (copias locales de clips catalogados)
configs_generados/              (JSON configs de videos generados)
catalogo_clips.json             (catálogo master de clips)
```

---

## Dependencias Principales
- selenium + webdriver-manager (scraping)
- instaloader (queries IG sin login)
- openai (Vision GPT-4o + text GPT-4o-mini)
- moviepy 2.0 + Pillow + numpy (video generation)
- python-dotenv (credenciales)
- ffmpeg (extracción de frames, externo)

---

## Limitaciones Conocidas (V1)
1. **Login manual** en paso 1 (Selenium no automatiza el login de IG)
2. **Perfiles hardcoded** (no config.json dinámico)
3. **Paso 4 es interactivo** (no puede correr desatendido)
4. **Pasos 5 y 6 no implementados** (el caption se decide en paso 4)
5. **No hay upload automático** a redes
6. **Single-threaded** todo el pipeline
7. **Solo Windows** testeado (paths de Brave, ffmpeg, etc.)
8. **Rate limiting conservador** en paso 2 (50 posts/sesión máximo)
9. **Catálogo de clips pequeño** (aún se está llenando manualmente)
10. **No hay retry automático** si un paso falla (main.py solo reporta)

# Propuestas de Mejora — Meme Reaction V2

Organizadas de menor a mayor esfuerzo, agrupadas por categoría.

---

## A. Optimizaciones del Pipeline Actual (Quick Wins)

### A1. Config centralizado (`config.json`)
- Mover TODOS los valores hardcoded a un JSON: perfiles_target, scroll_count, min_likes, max_por_sesion, delays, modelo IA, categorías
- Permite cambiar comportamiento sin tocar código
- Incluir flag `dry_run` global

### A2. Logging estructurado
- Reemplazar `print()` por `logging` con niveles (DEBUG/INFO/WARNING/ERROR)
- Archivo de log rotativo por sesión (`logs/run_2026-05-30_143000.log`)
- Métricas por run: tiempo por paso, tokens gastados, posts procesados

### A3. Retry automático con backoff exponencial
- En paso 2: si falla un post por timeout, reintentar 2x con delay creciente
- En paso 3: si OpenAI responde mal JSON, reintentar con temperatura más baja
- En main.py: opción `--retry-failed` que solo re-ejecuta pasos que fallaron

### A4. Paralelismo en clasificación (Paso 3)
- OpenAI soporta requests concurrentes. Usar `asyncio` + `aiohttp` para clasificar 3-5 memes simultáneos
- Reducción de tiempo: de ~2s/meme secuencial a ~0.5s/meme efectivo
- Respetar rate limits de la API (RPM/TPM)

### A5. Cache inteligente de clasificaciones
- Si un meme ya fue clasificado y el archivo no cambió (hash), no re-clasificar
- Útil cuando se hace `--redo-all` por cambio de prompt (solo re-hace los que el prompt nuevo afectaría)

---

## B. Nuevos Scripts / Módulos

### B1. `9_upload_social.py` — Upload automático
- Subir a TikTok, Instagram Reels, YouTube Shorts
- Usar APIs oficiales donde existan (TikTok Creator API, YouTube Data API v3)
- Para IG: usar instagrapi (ya la tienes como dependencia)
- Scheduling: poder programar hora de publicación
- Metadata: hashtags, descripción, música (donde aplique)

### B2. `0_scheduler.py` — Orquestador con cron/schedule
- Ejecutar el pipeline completo en horarios definidos (ej: scrape cada 6h, generar videos 1x/día)
- Usar `schedule` library o integrar con Task Scheduler de Windows
- Notificaciones (Telegram/Discord bot) cuando hay videos listos

### B3. `analytics.py` — Dashboard de métricas
- Cuántos memes scrapeados/descargados/clasificados/generados por día
- Hit rate: % de memes que pasan todos los filtros hasta video final
- Engagement predictor: correlacionar categorías con performance real post-upload
- Visualización simple con matplotlib o exportar a CSV

### B4. `clip_finder.py` — Descarga automática de clips de reacción
- Cuando el paso 4 sugiere un clip ideal que NO está en el catálogo, buscarlo automáticamente
- Fuentes: YouTube (yt-dlp), archivos locales, repositorios de clips virales
- IA describe el clip descargado y lo cataloga automáticamente

### B5. `batch_review.py` — Review en grid visual
- En vez de abrir imagen por imagen, mostrar un grid (Tkinter o HTML) con thumbnails
- Click para aprobar/rechazar. Mucho más rápido que el flujo actual.
- Alternativa: generar una página HTML estática con todas las imágenes y checkboxes

---

## C. Cambios Arquitecturales

### C1. Eliminar Selenium — Scraping headless con API
- **Opción A**: Usar Apify/Bright Data para scraping de IG (más estable, anti-ban)
- **Opción B**: Usar instagrapi para obtener posts del feed (ya tienes login)
- **Opción C**: RSS feeds de perfiles (existen servicios como Bibliogram)
- **Beneficio**: Elimina la dependencia de login manual y Brave instalado

### C2. Pipeline event-driven (no secuencial)
- Actualmente: main.py ejecuta 1→2→3→4→5→6→7→8 en orden
- Propuesta: Cada paso es independiente con triggers:
  - Paso 1 corre cada 6h automáticamente
  - Paso 2 se dispara cuando hay nuevos links
  - Paso 3 se dispara cuando hay nuevas imágenes
  - Paso 4 espera intervención humana (o se automatiza con C3)
  - Paso 7 genera cuando hay matches confirmados
- Implementar con watchdog (file watchers) o una cola simple (JSON-based queue)

### C3. Automatizar Paso 4 al 100% (eliminar interactividad)
- Ahora: la IA sugiere y TÚ decides
- V2: Si accuracy > 70% Y hay clip en catálogo → auto-accept
- V2: Si accuracy < 40% → auto-skip (marcar para buscar clip)
- V2: Entre 40-70% → cola de "revisión humana" (pero no bloquea el pipeline)
- Esto permite que el pipeline corra 100% desatendido para los casos claros

### C4. Multi-perfil / Multi-plataforma scraping
- Agregar soporte para X/Twitter memes, Reddit (r/memes, r/dankmemes), 9GAG
- Cada fuente tiene su scraper (módulo pluggable)
- Normalizar output a formato estándar: `{image_url, source, engagement_metrics, shortcode}`

### C5. Base de datos SQLite en vez de JSONs
- Los JSONs actuales crecen linealmente y son frágiles (corrupción si crash mid-write)
- SQLite: queries complejas, integridad, concurrencia
- Tablas: memes, clips, matches, videos, uploads, metrics
- Permite analytics mucho más ricos y recovery de fallos

### C6. Modelo local para clasificación (ahorro de tokens)
- GPT-4o Vision cuesta ~$0.003/imagen (detail:low). A escala: 100 memes/día = $9/mes
- Alternativa: LLaVA / BLIP-2 local para clasificación básica
- Usar GPT-4o solo para casos ambiguos o para el caption creativo
- Hybrid: modelo local filtra invalidos + extrae texto → GPT-4o solo para ideas creativas

---

## D. Mejoras de UX / Calidad de Video

### D1. Templates de video configurables
- Actualmente: siempre meme arriba + clip abajo (fijo)
- V2: Múltiples templates: split horizontal, picture-in-picture, meme con borde de color, fade-in/out
- Selección automática basada en aspect ratio del meme y categoría

### D2. Transiciones y efectos
- Zoom lento en el meme (Ken Burns effect) para que no sea estático
- Shake effect cuando el clip tiene reacción fuerte
- Fade-in del caption con timing sincronizado al clip

### D3. Audio inteligente
- Actualmente: SIEMPRE audio del clip (restricción)
- V2: Librería de audios de fondo por categoría (sad, hype, suspense)
- Opción de overlay: audio del clip + música de fondo baja
- Detección de silencio en el clip → agregar SFX (risa, suspense, etc.)

### D4. A/B Testing de thumbnails
- Generar 2-3 variantes del mismo video (diferente caption, diferente clip)
- Subir la mejor según engagement de los primeros minutos
- Requiere el módulo de upload + analytics

### D5. Caption inteligente con timing
- Actualmente: caption estático durante todo el video
- V2: Caption que aparece DESPUÉS de que el viewer lee el meme (delay de 1-2s)
- O caption que aparece palabra por palabra (typewriter effect)
- Sincronizar con momento clave del clip (la "reacción")

---

## E. Seguridad y Mantenimiento

### E1. Rate limit budget system
- Trackear cuántos requests se hicieron a IG y OpenAI por día/semana
- Alertar cuando se acerca al límite
- Auto-pausar si se excede (en vez de esperar el ban)

### E2. Health checks y self-healing
- Antes de ejecutar: verificar que Brave existe, ffmpeg existe, .env tiene keys, IG no está baneado
- Si algo falla: mensaje claro de qué arreglar
- Auto-cleanup de archivos temporales (`_temp_video/`)

### E3. Versionado de prompts
- Guardar cada versión del prompt de clasificación con su fecha
- Poder comparar accuracy de clasificación entre versiones
- Rollback si una versión nueva es peor

### E4. Backup automático de historiales
- Los JSONs de historial son la "memoria" del proyecto
- Backup diario a otra carpeta / Google Drive / S3
- O moverlos a SQLite (ver C5)

---

## Priorización Sugerida para V2

| Prioridad | Mejora | Impacto | Esfuerzo |
| --- | --- | --- | --- |
| 1 | A1 (config.json) | Alto | Bajo |
| 2 | C3 (auto paso 4) | Muy Alto | Medio |
| 3 | C1 (eliminar Selenium) | Alto | Medio |
| 4 | B1 (upload social) | Muy Alto | Alto |
| 5 | A4 (paralelismo paso 3) | Medio | Bajo |
| 6 | C5 (SQLite) | Alto | Medio |
| 7 | D1 (templates video) | Alto | Medio |
| 8 | B2 (scheduler) | Alto | Medio |
| 9 | D5 (caption timing) | Medio | Medio |
| 10 | C4 (multi-plataforma) | Alto | Alto |